# 14z Verify MVDeTr Detector MODA
CPU-only detector verifier for the 12a WILDTRACK MVDeTr `test.txt` artifact. It evaluates detector-only ground-plane MODA 0.921 +/- 0.005 without training, inference, tracking, ReID, or association stages.

In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/MRKDaGods/gp.git"
EXPECTED_MASTER_SHA_AT_BUILD = "e67cd24076e66dadb5aff33b46a514a0514caa7d"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"
RESULTS_PATH = WORK_DIR / "14z_verify_results.json"

def run(cmd, *, cwd=None, check=True, capture=False):
    print("$", " ".join(map(str, cmd)))
    if capture:
        return subprocess.check_output(list(map(str, cmd)), cwd=cwd, text=True)
    return subprocess.run(list(map(str, cmd)), cwd=cwd, check=check)

def pip_install(*args):
    run([sys.executable, "-m", "pip", "install", "-q", *args])

if not PROJECT.exists():
    run(["git", "clone", "--branch", "master", "--depth", "1", REPO_URL, str(PROJECT)])
else:
    run(["git", "-C", PROJECT, "fetch", "origin", "master", "--depth", "1"])
    run(["git", "-C", PROJECT, "checkout", "master"])
    run(["git", "-C", PROJECT, "reset", "--hard", "origin/master"])

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
head_sha = run(["git", "rev-parse", "HEAD"], capture=True).strip()
print("resolved git SHA:", head_sha)
print("expected master SHA at notebook build:", EXPECTED_MASTER_SHA_AT_BUILD)
if head_sha != EXPECTED_MASTER_SHA_AT_BUILD:
    print(f"WARN: master moved since notebook build (expected {EXPECTED_MASTER_SHA_AT_BUILD}, got {head_sha}); proceeding because metric assertions are authoritative.")
print("python:", sys.version)
print("platform:", platform.platform())

pip_install("faiss-cpu", "motmetrics", "loguru", "omegaconf", "rich", "networkx>=3.1", "click")
pip_install("filterpy", "ftfy", "lapx", "timm")
pip_install("--no-deps", "ultralytics")
pip_install("--no-deps", "boxmot==11.0.3")
pip_install("--no-deps", "-e", ".")

help_text = run([sys.executable, "scripts/run_pipeline.py", "--help"], capture=True)
RUN_ID_SHIM_APPLIED = False
if "--run-id" not in help_text:
    script_path = Path("scripts/run_pipeline.py")
    script_text = script_path.read_text(encoding="utf-8")
    script_text = script_text.replace(
        '@click.option("--dry-run", is_flag=True, default=False, help="Print resolved plan without running stages")\n',
        '@click.option("--dry-run", is_flag=True, default=False, help="Print resolved plan without running stages")\n'
        '@click.option("--run-id", default=None, help="Run id/name for the output directory")\n',
    )
    script_text = script_text.replace(
        'def main(config: str, dataset_config: str, stages: str, smoke_test: bool, dry_run: bool, override: tuple):',
        'def main(config: str, dataset_config: str, stages: str, smoke_test: bool, dry_run: bool, run_id: str | None, override: tuple):',
    )
    script_text = script_text.replace(
        '    if apply_cpu_when_no_cuda(cfg):\n',
        '    if run_id:\n        cfg.project.run_name = run_id\n\n    if apply_cpu_when_no_cuda(cfg):\n',
    )
    script_path.write_text(script_text, encoding="utf-8")
    RUN_ID_SHIM_APPLIED = True
    print("Applied runtime-only --run-id compatibility shim")
else:
    print("run_pipeline.py already exposes --run-id")

for mount in [Path("/tmp"), WORK_DIR]:
    total, used, free = shutil.disk_usage(mount)
    print(f"{mount}: {free / 1024**3:.1f} GiB free / {total / 1024**3:.1f} GiB total")

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import tarfile
import time

INPUT_ROOT = Path("/kaggle/input")

def write_json(path: Path, payload: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True), encoding="utf-8")
    print(f"wrote {path}")

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def symlink_or_copy(src: Path, dst: Path) -> None:
    if dst.exists() or dst.is_symlink():
        if dst.is_dir() and not dst.is_symlink():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        dst.symlink_to(src, target_is_directory=src.is_dir())
    except OSError:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)

def list_input_roots(max_items: int = 80) -> list[str]:
    roots = sorted(str(path) for path in INPUT_ROOT.iterdir()) if INPUT_ROOT.exists() else []
    for item in roots[:max_items]:
        print(item)
    if len(roots) > max_items:
        print(f"... {len(roots) - max_items} more")
    return roots

def find_input_dir(owner_slug: str, *hints: str) -> Path | None:
    owner, slug = owner_slug.split("/", 1)
    direct_candidates = [
        INPUT_ROOT / slug,
        INPUT_ROOT / owner_slug.replace("/", "-"),
        INPUT_ROOT / "notebooks" / owner / slug,
        INPUT_ROOT / "datasets" / owner / slug,
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate
    wanted = [slug.lower(), *(hint.lower() for hint in hints)]
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir():
            text = str(path).lower()
            if all(token in text for token in wanted if token):
                return path
    return None

def find_file(names: list[str], *, owner_slug: str | None = None, hints: tuple[str, ...] = ()) -> Path:
    roots = []
    if owner_slug:
        root = find_input_dir(owner_slug, *hints)
        if root is not None:
            roots.append(root)
    roots.append(INPUT_ROOT)
    matches = []
    lowered_hints = tuple(h.lower() for h in hints)
    for root in roots:
        if root is None or not root.exists():
            continue
        for name in names:
            matches.extend(root.rglob(name))
    if owner_slug or hints:
        filtered = []
        for path in matches:
            text = str(path).lower()
            if owner_slug and owner_slug.split("/", 1)[1].lower() in text:
                filtered.append(path)
            elif lowered_hints and all(h in text for h in lowered_hints):
                filtered.append(path)
        matches = filtered or matches
    matches = sorted(set(matches), key=lambda p: (len(str(p)), str(p)))
    if not matches:
        raise FileNotFoundError(f"Could not find {names} owner_slug={owner_slug} hints={hints}")
    print("selected", matches[0])
    return matches[0]

def find_wildtrack_root() -> Path:
    markers = {"Image_subsets", "annotations_positions", "calibrations"}
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir():
            children = {child.name for child in path.iterdir() if child.is_dir()}
            if markers.issubset(children):
                print("WILDTRACK root:", path)
                return path
    raise FileNotFoundError("Could not find WILDTRACK root with Image_subsets, annotations_positions, calibrations")

def find_cityflow_root() -> Path:
    candidates = [
        INPUT_ROOT / "data-aicity-2023-track-2",
        INPUT_ROOT / "datasets" / "thanhnguyenle" / "data-aicity-2023-track-2",
    ]
    for root in candidates:
        if root.exists():
            return root
    for path in INPUT_ROOT.rglob("*") if INPUT_ROOT.exists() else []:
        if path.is_dir() and ((path / "train").exists() or (path / "AIC22_Track1_MTMC_Tracking").exists()):
            return path
    raise FileNotFoundError("Could not find CityFlowV2/AIC22 dataset root")

def prepare_cityflow_layout(source_root: Path) -> Path:
    raw_parent = PROJECT / "data" / "raw"
    tmp_parent = Path("/tmp/datasets")
    tmp_parent.mkdir(parents=True, exist_ok=True)
    if raw_parent.exists() or raw_parent.is_symlink():
        if raw_parent.is_symlink() or raw_parent.is_file():
            raw_parent.unlink()
        else:
            shutil.rmtree(raw_parent)
    raw_parent.parent.mkdir(parents=True, exist_ok=True)
    raw_parent.symlink_to(tmp_parent, target_is_directory=True)
    data_raw = tmp_parent / "cityflowv2"
    data_raw.mkdir(parents=True, exist_ok=True)
    roots = []
    if (source_root / "AIC22_Track1_MTMC_Tracking").exists():
        roots.append(source_root / "AIC22_Track1_MTMC_Tracking")
    roots.append(source_root)
    for root in roots:
        for split in ["train", "validation", "test"]:
            split_dir = root / split
            if not split_dir.exists():
                continue
            for scene_dir in sorted(split_dir.iterdir()):
                if not scene_dir.is_dir():
                    continue
                for cam_dir in sorted(scene_dir.iterdir()):
                    if not cam_dir.is_dir():
                        continue
                    flat = data_raw / f"{scene_dir.name}_{cam_dir.name}"
                    if not flat.exists():
                        symlink_or_copy(cam_dir, flat)
    expected = ["S01_c001", "S01_c002", "S01_c003", "S02_c006", "S02_c007", "S02_c008"]
    missing = [cam for cam in expected if not (data_raw / cam / "gt" / "gt.txt").exists()]
    assert not missing, f"CityFlow GT camera layout missing: {missing} under {data_raw}"
    print("CityFlow layout ready:", data_raw)
    return data_raw

In [ ]:
list_input_roots()
WILDTRACK_ROOT = find_wildtrack_root()
ANNOTATIONS_DIR = WILDTRACK_ROOT / "annotations_positions"
CALIBRATIONS_DIR = WILDTRACK_ROOT / "calibrations"
TEST_TXT = find_file(["test.txt"], owner_slug="yahiaakhalafallah/12a-resume-emit-wildtrack-test-txt", hints=("12a", "resume", "wildtrack", "mvdetr"))
first_lines = []
with TEST_TXT.open("r", encoding="utf-8") as fh:
    for line in fh:
        if line.strip():
            first_lines.append(line.strip())
        if len(first_lines) >= 20:
            break
for line in first_lines[:5]:
    parts = line.split()
    assert len(parts) >= 3
    [float(parts[i]) for i in range(3)]
    print(line)
line_count = sum(1 for line in TEST_TXT.open("r", encoding="utf-8") if line.strip())
print("test.txt", TEST_TXT, "lines", line_count, "sha256", sha256_file(TEST_TXT))
print("calibrations", CALIBRATIONS_DIR)

In [ ]:
from src.stage_wildtrack_mvdetr.pipeline import load_mvdetr_ground_plane_detections
from src.stage5_evaluation.ground_plane_eval import evaluate_ground_plane, load_gt_ground_positions

detections = load_mvdetr_ground_plane_detections(TEST_TXT, normalize_wildtrack_frames=True)
assert detections, "no MVDeTr detections loaded"
gt = load_gt_ground_positions(ANNOTATIONS_DIR)
assert gt, "no WILDTRACK ground-truth positions loaded"
pred = {}
for pred_id, det in enumerate(detections, start=1):
    pred.setdefault(det.frame_id, []).append((pred_id, det.x_cm, det.y_cm))
det_frames = sorted({det.frame_id for det in detections})
gt_frames = sorted(gt)
gt_eval = {frame_id: gt[frame_id] for frame_id in det_frames if frame_id in gt}
gt_eval_frames = sorted(gt_eval)
print("detections", len(detections), "frames", det_frames[0], det_frames[-1])
print("gt frames", len(gt_frames), gt_frames[0], gt_frames[-1])
print("eval gt frames", len(gt_eval_frames), gt_eval_frames[0], gt_eval_frames[-1])
assert det_frames == list(range(40)), "test.txt should normalize to the 40-frame WILDTRACK test split (0..39)"
assert gt_eval_frames == det_frames, "normalized GT frames do not match detection frames"
metrics = evaluate_ground_plane(gt_eval, pred, threshold_cm=50.0)
moda = float(metrics["moda"])
delta = moda - 0.921
status = "pass" if abs(delta) <= 0.005 else "fail"
result = {
    "verifier": "14z_verify_mvdetr_detector",
    "status": status,
    "repo_sha": head_sha,
    "run_id_shim_applied": RUN_ID_SHIM_APPLIED,
    "target_moda": 0.921,
    "actual_moda": moda,
    "delta": delta,
    "tolerance": 0.005,
    "modp": metrics.get("modp"),
    "precision": metrics.get("precision"),
    "recall": metrics.get("recall"),
    "false_positives": metrics.get("false_positives"),
    "misses": metrics.get("misses"),
    "id_switches": metrics.get("id_switches"),
    "evaluation_path": "detector_only_evaluate_ground_plane",
    "gt_frame_count": len(gt_frames),
    "evaluated_gt_frame_count": len(gt_eval_frames),
    "evaluated_frame_min": gt_eval_frames[0],
    "evaluated_frame_max": gt_eval_frames[-1],
    "test_txt_path": str(TEST_TXT),
    "test_txt_sha256": sha256_file(TEST_TXT),
    "line_count": line_count,
    "wildtrack_root": str(WILDTRACK_ROOT),
}
write_json(RESULTS_PATH, result)
print(json.dumps(result, indent=2))
assert abs(delta) <= 0.005